# HeliOS LSTM — Documentação do Treinamento Multi-Plataforma
**HeliOS | Global Solution 2026.1 — FIAP | 2º Ano TIAO**

---

Este notebook arquiva o fluxo completo do treinamento da rede LSTM do HeliOS, documentando:
- ⚠️ **Impedimento**: congelamento na Época 1 em macOS ARM com TensorFlow 2.21 + Python 3.13
- ✅ **Solução**: treinamento executado no Google Colab (GPU T4) em ~5 minutos
- 📦 **Artefatos**: validação dos arquivos gerados e inferência end-to-end

**Arquivos do modelo**: `src/ml/lstm/model/`  
**Script Colab portátil**: `src/ml/lstm/colab_train.py`  
**Plano de implementação**: `PLANO_IMPLEMENTACAO_HELIOS.md`

## 1. Detecção do Ambiente e Verificação de GPU

In [ ]:
import os
import sys
import platform

# Detecta se está rodando no Google Colab ou localmente
IN_COLAB = "google.colab" in sys.modules
runtime = "Google Colab" if IN_COLAB else f"Local ({platform.system()} {platform.machine()})"

print(f"Ambiente de execução : {runtime}")
print(f"Python               : {sys.version.split()[0]}")
print(f"Sistema operacional  : {platform.platform()}")

import tensorflow as tf
print(f"\nTensorFlow           : {tf.__version__}")

gpus = tf.config.list_physical_devices("GPU")
cpus = tf.config.list_physical_devices("CPU")
print(f"GPUs disponíveis     : {len(gpus)} — {[g.name for g in gpus] if gpus else 'nenhuma'}")
print(f"CPUs disponíveis     : {len(cpus)}")

if not IN_COLAB:
    print("\n⚠️  Ambiente LOCAL — modelo pré-treinado via Google Colab. Veja Seção 7 para o histórico.")
else:
    print("\n✅  Ambiente COLAB — GPU disponível para treinamento.")

## 2. Carregamento dos Metadados do Modelo Treinado

In [ ]:
import json
from pathlib import Path

# Caminho relativo ao notebook (src/ml/lstm/)
NOTEBOOK_DIR = Path(".")
MODEL_DIR = NOTEBOOK_DIR / "model"
ROOT = NOTEBOOK_DIR.parent.parent.parent  # raiz do projeto

meta_path = MODEL_DIR / "model_meta.json"
with open(meta_path) as f:
    meta = json.load(f)

print("=" * 50)
print("  HeliOS LSTM — Metadados do Modelo")
print("=" * 50)
print(f"  Janela de entrada  : {meta['window_size']} meses")
print(f"  Horizonte de saída : {meta['horizon']} meses")
print(f"  Épocas treinadas   : {meta['epochs_trained']}")
print(f"  MAE (teste)        : {meta['mae_test']:.4f} manchas solares")
print(f"  RMSE (teste)       : {meta['rmse_test']:.4f} manchas solares")
print(f"  Feature            : {meta['feature'].upper()} (Sunspot Number)")
print(f"  Treinado em        : {meta.get('trained_on', 'N/A')}")
print(f"  Meses usados       : últimos {meta.get('last_n_months', 'N/A')} (~{meta.get('last_n_months', 0)//12} anos)")

## 3. Validação dos Artefatos em `src/ml/lstm/model/`

In [ ]:
import datetime

expected_files = [
    "helios_lstm.keras",
    "scaler.pkl",
    "model_meta.json",
]

print(f"{'Arquivo':<30} {'Tamanho':>12}  {'Modificado em'}")
print("-" * 70)
all_ok = True
for fname in expected_files:
    fpath = MODEL_DIR / fname
    if fpath.exists():
        size_kb = fpath.stat().st_size / 1024
        mtime = datetime.datetime.fromtimestamp(fpath.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S")
        print(f"  ✅  {fname:<26} {size_kb:>9.1f} KB  {mtime}")
    else:
        print(f"  ❌  {fname:<26} {'NÃO ENCONTRADO':>12}")
        all_ok = False

print()
if all_ok:
    print("Todos os artefatos estão presentes e prontos para uso.")
else:
    print("⚠️  Artefatos faltando — execute o treinamento via colab_train.py")

## 4. Carregar Modelo e Executar Inferência End-to-End

In [ ]:
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler

# Caminhos — ajusta se executar de outro diretório
DATA_PATH = Path("../../../data/processed/solar_cycle_historical.csv")

# 1. Carregar modelo e scaler
print("Carregando modelo e scaler...")
model = tf.keras.models.load_model(str(MODEL_DIR / "helios_lstm.keras"))
with open(MODEL_DIR / "scaler.pkl", "rb") as f:
    scaler: MinMaxScaler = pickle.load(f)

WINDOW = meta["window_size"]
HORIZON = meta["horizon"]

# 2. Preparar janela de entrada com os dados mais recentes
df = pd.read_csv(DATA_PATH)
df["ssn"] = df["ssn"].fillna(0).clip(lower=0).astype(float)
series = df["ssn"].values[-600:].astype(float)  # últimos 600 meses (consistente com treino)

series_norm = scaler.transform(series.reshape(-1, 1)).flatten()
window_input = series_norm[-WINDOW:].reshape(1, WINDOW, 1)  # última janela disponível

# 3. Inferência
pred_norm = model.predict(window_input, verbose=0)
pred_ssn = scaler.inverse_transform(pred_norm)[0]

# 4. Classificação de risco por nível de SSN
def classify_risk(ssn: float) -> str:
    if ssn < 50:   return "QUIET"
    if ssn < 80:   return "ACTIVE"
    if ssn < 120:  return "ELEVATED"
    return "STORM"

print(f"\nPrevisão de SSN para os próximos {HORIZON} meses (a partir de 2026-06):")
print(f"{'Mês':<8} {'SSN Previsto':>14} {'Risco':>12}")
print("-" * 38)
for i, ssn in enumerate(pred_ssn, 1):
    risk = classify_risk(ssn)
    print(f"  +{i:<5}  {ssn:>12.1f}   {risk:>10}")

print(f"\n✅ Modelo carregado e inferência funcionando corretamente.")

## 5. Reprodução das Métricas MAE e RMSE

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Recria o split temporal idêntico ao usado no treino
series_600 = df["ssn"].values[-600:].astype(float)
series_norm_600 = scaler.transform(series_600.reshape(-1, 1)).flatten()

X_all, y_all = [], []
for i in range(len(series_norm_600) - WINDOW - HORIZON + 1):
    X_all.append(series_norm_600[i : i + WINDOW])
    y_all.append(series_norm_600[i + WINDOW : i + WINDOW + HORIZON])
X_all = np.array(X_all).reshape(-1, WINDOW, 1)
y_all = np.array(y_all)

n = len(X_all)
val_end = int(n * 0.90)
X_test = X_all[val_end:]
y_test = y_all[val_end:]

y_pred_norm = model.predict(X_test, verbose=0)
y_pred = scaler.inverse_transform(y_pred_norm)
y_true = scaler.inverse_transform(y_test)

mae_calc  = mean_absolute_error(y_true.flatten(), y_pred.flatten())
rmse_calc = np.sqrt(mean_squared_error(y_true.flatten(), y_pred.flatten()))

print(f"{'Métrica':<10}  {'model_meta.json':>18}  {'Calculado agora':>18}  {'Match?':>8}")
print("-" * 62)
print(f"  MAE    {meta['mae_test']:>18.4f}  {mae_calc:>18.4f}  {'✅' if abs(meta['mae_test'] - mae_calc) < 1.0 else '⚠️'}")
print(f"  RMSE   {meta['rmse_test']:>18.4f}  {rmse_calc:>18.4f}  {'✅' if abs(meta['rmse_test'] - rmse_calc) < 1.0 else '⚠️'}")
print(f"\n  Amostras de teste: {len(X_test)}")
print(f"  Referência SSN médio histórico: ~82 | Máximo ciclo solar 25: ~211")

## 6. Curvas de Treinamento (PNG gerado no Colab)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# O PNG foi gerado no Colab e baixado junto com o modelo
png_path = Path("../../../docs/lstm_training_metrics.png")

if png_path.exists():
    img = mpimg.imread(str(png_path))
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(
        f"HeliOS LSTM — Curvas de Treinamento (Colab T4 GPU)\n"
        f"MAE={meta['mae_test']:.2f} SSN | RMSE={meta['rmse_test']:.2f} SSN | "
        f"Épocas={meta['epochs_trained']}",
        fontsize=12,
    )
    plt.tight_layout()
    plt.show()
else:
    print(f"PNG não encontrado em: {png_path}")
    print("Copie lstm_training_metrics.png do Colab para docs/")

## 7. Registro do Impedimento — Congelamento Local (macOS ARM + TF 2.21)

### Cronologia do impedimento

| Data | Evento |
|------|--------|
| 2026-06-05 14:05 | Primeira tentativa de treino local (80 épocas, 3329 amostras) — Epoch 1/80 congelou |
| 2026-06-05 14:25 | Abort após 20 min. Tentativa 2 com `--fast` (50 épocas, 600 amostras) — Epoch 1/50 congelou |
| 2026-06-05 14:45 | Adicionadas flags `TF_XLA_FLAGS`, `jit_compile=False` — sem efeito |
| 2026-06-05 15:00 | Decisão: externalizar treinamento para Google Colab |
| 2026-06-05 15:05 | Treinamento concluído no Colab (GPU T4) em ~5 minutos |
| 2026-06-05 15:10 | Artefatos baixados e colocados em `src/ml/lstm/model/` |

### Diagnóstico técnico
- **Ambiente afetado**: macOS 14 Apple Silicon (ARM64), Python 3.13, TensorFlow 2.21+
- **Causa raiz**: XLA/JIT compilation na Época 1 entra em loop aguardando aceleradores (CUDA/Metal) que não estão disponíveis nesta combinação
- **Mitigações tentadas**: `TF_XLA_FLAGS=--tf_xla_auto_jit=0`, `tf.config.optimizer.set_jit(False)`, `jit_compile=False` no `model.compile()` — todas sem efeito
- **Solução definitiva**: Google Colab com GPU T4 via `src/ml/lstm/colab_train.py`

### Recomendação para futuras execuções
Qualquer retreinamento deve ser feito via `colab_train.py` no Google Colab, ou em ambiente Linux com TensorFlow ≤ 2.15 (compatível com macOS ARM).

## 8. Status do Plano de Implementação — Fase 5

In [ ]:
from datetime import datetime

phase5_tasks = [
    {"tarefa": "Criar src/ml/lstm/train.py",         "status": "DONE",    "plataforma": "Local (VS Code)"},
    {"tarefa": "Criar src/ml/lstm/evaluate.py",       "status": "DONE",    "plataforma": "Local (VS Code)"},
    {"tarefa": "Criar src/ml/lstm/predict.py",        "status": "DONE",    "plataforma": "Local (VS Code)"},
    {"tarefa": "Criar src/ml/lstm/colab_train.py",    "status": "DONE",    "plataforma": "Local (VS Code)"},
    {"tarefa": "Treinar modelo (tentativa local)",    "status": "BLOCKED", "plataforma": "macOS ARM — XLA freeze"},
    {"tarefa": "Treinar modelo (Google Colab GPU)",   "status": "DONE",    "plataforma": "Google Colab T4 GPU"},
    {"tarefa": "helios_lstm.keras",                   "status": "DONE",    "plataforma": "src/ml/lstm/model/"},
    {"tarefa": "scaler.pkl",                          "status": "DONE",    "plataforma": "src/ml/lstm/model/"},
    {"tarefa": "model_meta.json",                     "status": "DONE",    "plataforma": "src/ml/lstm/model/"},
    {"tarefa": "docs/lstm_training_metrics.png",      "status": "DONE",    "plataforma": "docs/"},
    {"tarefa": "Upload modelo para S3",               "status": "PENDING", "plataforma": "AWS S3 helios-solar-data"},
    {"tarefa": "Commit e push Fase 5",                "status": "PENDING", "plataforma": "GitHub"},
]

status_icons = {"DONE": "✅", "BLOCKED": "🚫", "PENDING": "⬜"}

print(f"  HeliOS — Fase 5 (LSTM) — Status em {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("=" * 72)
print(f"  {'Tarefa':<45} {'Status':>8}  {'Detalhe'}")
print("-" * 72)
for t in phase5_tasks:
    icon = status_icons[t["status"]]
    print(f"  {icon} {t['tarefa']:<43} {t['status']:>8}  {t['plataforma']}")

done  = sum(1 for t in phase5_tasks if t["status"] == "DONE")
total = len(phase5_tasks)
print(f"\n  Progresso: {done}/{total} tarefas concluídas ({done/total*100:.0f}%)")